Case #1: Orders not Delivered

**Brief Case:**

The WideWorldImporters warehouse has been receiving reports from customers claiming theu never received their order. This has been causing the company thousands of dollars in loses. Your job is to track down which orders have not been confirmed as delivered and identify why the delivery failed.

**Objectives:**

1. Get a list of orders that do not have a confirmed delivery time
2. Find similarities between those orders to narrow down why it had not been delivered

In [5]:
SELECT 
    i.InvoiceID,
    i.CustomerID,
    c.CustomerName,
    i.SalespersonPersonID,
    i.OrderID,
    i.InvoiceDate,
    i.DeliveryInstructions,
    e.CityName,
    s.StateProvinceName
FROM Sales.Invoices AS i
JOIN Sales.Customers AS c ON i.CustomerID = c.CustomerID
JOIN Application.Cities AS e ON c.DeliveryCityID = e.CityID
JOIN Application.StateProvinces AS s ON e.StateProvinceID = s.StateProvinceID
WHERE i.ConfirmedDeliveryTime IS NULL

(84 rows affected)

Total execution time: 00:00:00.112

InvoiceID,CustomerID,CustomerName,SalespersonPersonID,OrderID,InvoiceDate,DeliveryInstructions,CityName,StateProvinceName
70471,6,"Tailspin Toys (Jessie, ND)",14,73547,2016-05-31,"Shop 196, 483 Raut Lane",Jessie,North Dakota
70497,11,"Tailspin Toys (Devault, PA)",14,73573,2016-05-31,"Unit 250, 1432 Pullela Street",Devault,Pennsylvania
70443,28,"Tailspin Toys (North Ridge, NY)",15,73519,2016-05-31,"Unit 40, 890 Hlouskova Avenue",North Ridge,New York
70499,28,"Tailspin Toys (North Ridge, NY)",2,73575,2016-05-31,"Unit 40, 890 Hlouskova Avenue",North Ridge,New York
70481,29,"Tailspin Toys (Eulaton, AL)",8,73557,2016-05-31,"Shop 16, 1606 Ahmadian Road",Eulaton,Alabama
70431,35,"Tailspin Toys (Slanesville, WV)",14,73507,2016-05-31,"Suite 150, 1959 Sarma Road",Slanesville,West Virginia
70488,35,"Tailspin Toys (Slanesville, WV)",20,73564,2016-05-31,"Suite 150, 1959 Sarma Road",Slanesville,West Virginia
70485,64,"Tailspin Toys (Hodgdon, ME)",16,73561,2016-05-31,"Shop 133, 967 Alizadeh Boulevard",Hodgdon,Maine
70437,76,"Tailspin Toys (Yewed, OK)",7,73513,2016-05-31,"Unit 266, 210 safranek Lane",Yewed,Oklahoma
70462,82,"Tailspin Toys (La Cueva, NM)",15,73538,2016-05-31,"Suite 166, 1708 Ankitham Street",La Cueva,New Mexico


**Schemas Needed to Solve the Case with Table or View Examples:**

- Sales.Invoices
- Sales.Customers
- Sales.Orders
- Application.Cities
- Application.StateProvinces
- Application.DeliveryMethods

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Identify all orders with NULL ConfirmedDeliveryTime from the Sales.Invoices table
- Join with the customers and orders table to see which customers were affected
- Join with the cities, state provinces, and delivery methods tables to see if a specific location or delivery method caused the problem

Case #2: High-Risk Customers

**Brief Case:**

The WideWorldImporters warehouse has noticed several customers that have no credit limit assigned in the database. This could be a potential risk to the company, since those customers may have large unpaid invoices in the future. Your task is to identity the cusomers with no credit limit and determine if they are a high-risk customer.

**Objectives:**

1. Get a list of customers with no credit limit
2. Find other characteristics available in the database of the customers
3. Use the information found to determine if the customer is high-risk

In [10]:
SELECT 
    c.CustomerID,
    c.CustomerName,
    c.CreditLimit,
    SUM(t.OutstandingBalance) AS TotalOutstanding
FROM Sales.Customers AS c
LEFT JOIN Sales.CustomerTransactions AS t ON c.CustomerID = t.CustomerID
WHERE c.CreditLimit IS NULL
GROUP BY c.CustomerID, c.CustomerName, c.CreditLimit
ORDER BY TotalOutstanding DESC;

(402 rows affected)

Total execution time: 00:00:00.303

CustomerID,CustomerName,CreditLimit,TotalOutstanding
401,Wingtip Toys (Head Office),NULL,97053.58
1,Tailspin Toys (Head Office),NULL,56435.84
2,"Tailspin Toys (Sylvanite, MT)",NULL,NULL
3,"Tailspin Toys (Peeples Valley, AZ)",NULL,NULL
4,"Tailspin Toys (Medicine Lodge, KS)",NULL,NULL
5,"Tailspin Toys (Gasport, NY)",NULL,NULL
6,"Tailspin Toys (Jessie, ND)",NULL,NULL
7,"Tailspin Toys (Frankewing, TN)",NULL,NULL
8,"Tailspin Toys (Bow Mar, CO)",NULL,NULL
9,"Tailspin Toys (Netcong, NJ)",NULL,NULL


**Schemas Needed to Solve the Case with Table or View Examples:**

- Sales.Customers
- Sales.CustomerTransactions

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Identify all customers with a NULL credit limit from the Sales.Customers table
- Look at their outstanding balance and transactions from the Customers.Transaction table

Case #3: Inactive Customers

**Brief Case:**

The WideWorldImporters warehouse wants to figure out how to bring back old customers. The company has noticed that there has been some customers that have not placed an order in a long time. Your job is to identify which customers have not placed any orders in 2016.

**Objectives:**

1. Identify customers that have not placed an order after the given date
2. Identify patterns such as items purchased and region to understand why they have not made any recent purchases

In [18]:
SELECT 
    c.CustomerID,
    c.CustomerName,
    MAX(o.OrderDate) AS LastOrder
FROM Sales.Customers AS c
LEFT JOIN Sales.Orders AS o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName
HAVING MAX(o.OrderDate) < '2016-01-01'
ORDER BY LastOrder DESC;

(0 rows affected)

Total execution time: 00:00:00.114

CustomerID,CustomerName,LastOrder


**Schemas Needed to Solve the Case with Table or View Examples:**

- Sales.Orders
- Sales.Customers

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Start by identifying all customers that doesn't have an order date in 2016
- Look at their previous orders to find patterns of why they stopped making purchases

Case #4: Loyal Customers

**Brief Case:**

The WideWorldImporters warehouse wants to reward loyal customers who have made at least 20 orders every year. Your task is to identify any customers that fit the criteria.

**Objectives:**

1. Identify customers that have placed at least 20 orders every year

In [25]:
WITH CustomersList AS (
    SELECT 
        c.CustomerID,
        c.CustomerName,
        YEAR(o.OrderDate) AS OrderYear,
        COUNT(o.OrderID) AS OrderCount
    FROM Sales.Customers AS c
    JOIN Sales.Orders AS o ON c.CustomerID = o.CustomerID
    GROUP BY c.CustomerID, c.CustomerName, YEAR(o.OrderDate)
)

SELECT 
    CustomerID,
    CustomerName
FROM CustomersList
GROUP BY CustomerID, CustomerName
HAVING MIN(OrderCount) >= 20

(80 rows affected)

Total execution time: 00:00:00.156

CustomerID,CustomerName
5,"Tailspin Toys (Gasport, NY)"
16,"Tailspin Toys (Coney Island, MO)"
29,"Tailspin Toys (Eulaton, AL)"
32,"Tailspin Toys (Maypearl, TX)"
35,"Tailspin Toys (Slanesville, WV)"
37,"Tailspin Toys (Kerby, OR)"
42,"Tailspin Toys (Arietta, NY)"
45,"Tailspin Toys (Severna Park, MD)"
48,"Tailspin Toys (Trentwood, WA)"
49,"Tailspin Toys (Muir, MI)"


**Schemas Needed to Solve the Case with Table or View Examples:**

- Sales.Orders
- Sales.Customers

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Start by getting a list of customers that had at least 20 orders per year
- The database goes from 2013 to 2016, so make sure to check that range

Case #5: Sales Person Performance

**Brief Case:**

The WideWorldImporters warehouse wants to evaluate employee performance. They want to investigate any sales person who has not worked on 600 or more orders in a given time period. They want to focus on May 2015 to July 2015. Your job is to present a list of employees that for the criteria.

**Objectives:**

1. Identify the employees that were listed on less than 600 orders during the given months

In [29]:
SELECT 
    e.FullName AS EmployeeName,
    COUNT(o.OrderID) AS OrderCount
FROM Sales.Orders AS o
INNER JOIN Application.People e ON o.SalespersonPersonID = e.PersonID
WHERE o.OrderDate BETWEEN '2015-05-01' AND '2015-07-31'
GROUP BY e.FullName, e.PersonID
HAVING COUNT(o.OrderID) < 600
ORDER BY OrderCount DESC;

(2 rows affected)

Total execution time: 00:00:00.077

EmployeeName,OrderCount
Hudson Hollinworth,590
Anthony Grosse,587


**Schemas Needed to Solve the Case with Table or View Examples:**

- Sales.Orders
- Application.People

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Start by getting all the salespeople from the Application.People table
- Join with the Sales.Orders table to get the employee name with the corresponding orders
- Filter the dates to match the given date and add the criteria

Case #6: Employee Logins

**Brief Case:**

The WideWorldImporters warehouse is updating employee logins for security purposes. They want to make sure that every employee has a logon name and a password set up. Your job is to make sure that each employee has an account set up.

**Objectives:**

1. Identify if any employees are missing log in info

In [38]:
SELECT PersonID, FullName, LogonName, HashedPassword, IsEmployee
FROM Application.People AS e
WHERE (LogonName = 'NO LOGON') AND (HashedPassword IS NULL) AND IsEmployee = 1;

(0 rows affected)

Total execution time: 00:00:00.029

PersonID,FullName,LogonName,HashedPassword,IsEmployee


**Schemas Needed to Solve the Case with Table or View Examples:**

- <span style="color: var(--vscode-foreground);">Application.People</span>

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Start by indentifying all employees in the Application.People table
- Ensure that all employees have a logon name and a password set up

Case #7: Late Delivery Dilemma

**Brief Case:**

The WideWorldImporters warehouse wants to guarantee next day delivery for all its customers. It has been able to make that possible for many cusomers, but there were still multiple orders where it was not achieved. Your task is to identify orders where the difference between the order date and expected delivery date was more than one day and to indentify possible reasons why the delivery time was expected to be longer.

**Objectives:**

1. Identify orders that were expected to take more than one day to deliver
2. Find patterns in the orders to explain the shipping time

In [41]:
SELECT 
    o.OrderID, 
    c.CustomerID, 
    c.CustomerName,
    o.OrderDate, 
    o.ExpectedDeliveryDate,
    i.InvoiceID,
    i.InvoiceDate,
    i.DeliveryMethodID
FROM Sales.Orders AS o
JOIN Sales.Invoices AS i ON o.OrderID = i.OrderID
JOIN Sales.Customers AS c ON o.CustomerID = c.CustomerID
WHERE o.ExpectedDeliveryDate > DATEADD(DAY, 1, o.OrderDate)
ORDER BY o.ExpectedDeliveryDate DESC;

(18947 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.649

OrderID,CustomerID,CustomerName,OrderDate,ExpectedDeliveryDate,InvoiceID,InvoiceDate,DeliveryMethodID
73285,481,"Wingtip Toys (White Church, MO)",2016-05-27,2016-05-30,70227,2016-05-27,3
73286,853,Caterina Pinto,2016-05-27,2016-05-30,70228,2016-05-27,3
73287,16,"Tailspin Toys (Coney Island, MO)",2016-05-27,2016-05-30,70229,2016-05-27,3
73288,196,"Tailspin Toys (Howells, NE)",2016-05-27,2016-05-30,70230,2016-05-27,3
73289,42,"Tailspin Toys (Arietta, NY)",2016-05-27,2016-05-30,70231,2016-05-27,3
73290,1,Tailspin Toys (Head Office),2016-05-27,2016-05-30,70232,2016-05-27,3
73291,1031,Dipti Shah,2016-05-27,2016-05-30,70233,2016-05-27,3
73292,81,"Tailspin Toys (Big Moose, NY)",2016-05-27,2016-05-30,70234,2016-05-27,3
73293,832,Aakriti Byrraju,2016-05-27,2016-05-30,70235,2016-05-27,3
73294,996,Laszlo Gardenier,2016-05-27,2016-05-30,70236,2016-05-27,3


**Schemas Needed to Solve the Case with Table or View Examples:**

- Sales.Orders
- Sales.Customers
- Sales.Invoices

**Investigation Notes for Queries and Thoughts to Solve the Case:**

- Start by getting all the orders that took longer than one day to deliver
- Look at the customers and invoice table to compare different factors, such as delivery method and location, to identify why it was expected to take longer to deliver